# Árbol de decisión para Bank Marketing con Pipeline + MLflow

Este notebook entrena un modelo de árbol de decisión usando el dataset `bank.csv`.

La diferencia importante frente al notebook anterior es que ahora se guarda en MLflow un `Pipeline` completo:

```text
datos originales → preprocesamiento → árbol de decisión
```

De esta manera, Flask o Streamlit pueden enviar las columnas originales del dataset, sin tener que enviar columnas codificadas manualmente.


## 1. Importación de librerías

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from mlflow.models.signature import infer_signature

import mlflow
import mlflow.sklearn


D:\POSGRADO UTPL\Materias\Herramientas_IA\Clases\S6_MLFlow\entornoMLFlow\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración de MLflow



In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bank_arbol_pipeline")


2026/05/15 19:16:10 INFO mlflow.tracking.fluent: Experiment with name 'bank_arbol_pipeline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778890570583, experiment_id='1', last_update_time=1778890570583, lifecycle_stage='active', name='bank_arbol_pipeline', tags={}, trace_location=None, workspace='default'>

## 3. Carga del dataset

In [3]:
df = pd.read_csv("data/bank.csv", delimiter=";")
df.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [4]:
print("Filas y columnas:", df.shape)
df.info()


Filas y columnas: (4521, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


## 4. Preparación de datos

La variable objetivo `y` se transforma:

- `yes` → `1`
- `no` → `0`

Además, se elimina `duration` de las variables predictoras porque normalmente se conoce después de realizada la llamada. Por tanto, puede generar fuga de información si se usa para predecir antes de contactar al cliente.


In [5]:
df["y"] = df["y"].map({"yes": 1, "no": 0})

X = df.drop(columns=["y", "duration"])
y = df["y"]

print("Columnas originales usadas para entrenamiento:")
print(X.columns.tolist())

print("\nDistribución de la variable objetivo:")
print(y.value_counts())


Columnas originales usadas para entrenamiento:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']

Distribución de la variable objetivo:
y
0    4000
1     521
Name: count, dtype: int64


## 5. Identificación de columnas categóricas y numéricas

El pipeline hará automáticamente el `OneHotEncoder` para las columnas categóricas y dejará pasar las columnas numéricas.


In [6]:
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Columnas categóricas:")
print(columnas_categoricas)

print("\nColumnas numéricas:")
print(columnas_numericas)


Columnas categóricas:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Columnas numéricas:
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']


## 6. División de datos

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (3616, 15)
Prueba: (905, 15)


## 7. Creación del Pipeline

Este es el punto clave para producción.

El `Pipeline` contiene:

1. `preprocessor`: transforma variables categóricas con `OneHotEncoder`.
2. `modelo`: entrena el árbol de decisión.

Cuando guardamos este `Pipeline` en MLflow, Streamlit podrá enviar datos originales como `job`, `marital`, `education`, etc.


In [8]:
# Parámetros del árbol
criterion = "gini"
max_depth = 5
min_samples_split = 100
min_samples_leaf = 50
random_state = 42

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("modelo", DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        ))
    ]
)

pipeline


,steps,"[('preprocessor', ...), ('modelo', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 8. Entrenamiento, evaluación y registro en MLflow

El modelo se registrará con el nombre:

```text
clase06
```

Luego se podrá cargar desde Streamlit con:

```python
mlflow.sklearn.load_model("models:/arboles02/1")
```


In [9]:
with mlflow.start_run(run_name="arbol_pipeline_base") as run:

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Parámetros del algoritmo
    mlflow.log_param("modelo", "DecisionTreeClassifier")
    mlflow.log_param("criterion", criterion)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("random_state", random_state)

    # Parámetros del experimento
    mlflow.log_param("dataset", "bank.csv")
    mlflow.log_param("target", "y")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("stratify", True)
    mlflow.log_param("removed_column", "duration")
    mlflow.log_param("categorical_encoder", "OneHotEncoder")
    mlflow.log_param("handle_unknown", "ignore")

    # Métricas
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    
    # Firma del modelo: ayuda a documentar columnas esperadas
    input_example = X_train.head(5)
    signature = infer_signature(input_example, pipeline.predict(input_example))

    # Guardar y registrar el pipeline completo
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="modelo_bank_pipeline",
        registered_model_name="arboles02",
        input_example=input_example,
        signature=signature
    )

    print("RUN ID:", run.info.run_id)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("\nReporte de clasificación:")


D:\POSGRADO UTPL\Materias\Herramientas_IA\Clases\S6_MLFlow\entornoMLFlow\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/15 19:16:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 19:17:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cl

RUN ID: d1c3d4fc415d46aabf7a2fa7d894ba08
Accuracy: 0.8917127071823204
Precision: 0.6
Recall: 0.17307692307692307
F1: 0.26865671641791045

Reporte de clasificación:
🏃 View run arbol_pipeline_base at: http://127.0.0.1:5000/#/experiments/1/runs/d1c3d4fc415d46aabf7a2fa7d894ba08
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
